<a href="https://colab.research.google.com/github/pedroManuelP/C-digos-em-Python/blob/main/Lista2_do_Projeto_de_Sistemas_de_Controle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
import numpy as np
from numpy.polynomial import Polynomial
import re

def coef_amort(Mp_percent):
  # calcula o coeficiente de amortecimento a partir do overshoot em porcentagem
  log_mp = np.log(Mp_percent/100)
  return log_mp/np.sqrt(np.pi**2 + log_mp**2)

def freq_natural(ksi, tss, tss_percent):
  # calcula a frequência natural a partir do ksi, tss e a acomodação
  if(tss_percent == 5):
    constante = 3
  elif(tss_percent == 2):
    constante = 5
  return constante/(tss*ksi)

def sigma_angulos(s, pontos):
  sigma = 0
  for i in range(len(pontos)):
    sigma += np.rad2deg(np.atan2((s - pontos[i]).imag, (s - pontos[i]).real))
  return sigma

def phi_modulos(s, pontos):
  phi = 1
  for i in range(len(pontos)):
    phi *= np.abs((s - pontos[i]))
  return phi

def calculo_polos_dominante(s):
  if(s == '1'):
    print("Overshoot(%): ", end=''); Mp_percent = float(input())
    print("Tempo de acomodação(s): ", end=''); tss = float(input())
    print("Tolerância de acomodação(%): ", end=''); tss_percent = float(input())

    ksi = np.abs(coef_amort(Mp_percent)); ksi = np.round(ksi,4)
    wn = freq_natural(ksi,tss,tss_percent); wn = np.round(wn,4)
    print("Coeficiente de amortecimento: " + str(ksi))
    print("Frequência natural: " + str(wn) + " rad/s")

    wd = wn*np.sqrt(1-np.power(ksi,2)); wd = np.round(wd,4)
    print("Frequência amortecida: " + str(wd))

    s1 = np.round(complex(-ksi*wn,wd), 4); s2 = np.round(complex(-ksi*wn,-wd), 4)
  elif(s == '2'):
    print("Coeficiente de amortecimento: ", end = ''); ksi = float(input())
    print("Frequência natural: ", end = ''); wn = float(input())

    wd = wn*np.sqrt(1-np.power(ksi,2)); wd = np.round(wd,4)
    print("Frequência amortecida: " + str(wd))

    s1 = np.round(complex(-ksi*wn,wd), 4); s2 = np.round(complex(-ksi*wn,-wd), 4)
  elif(s == '3'):
    print("Parte real do polo: ", end=''); real = float(input())
    print("Parte imaginária do polo: ", end=''); imag = float(input())

    s1 = np.round(complex(real,imag), 4);
    s2 = np.round(complex(-real,-imag), 4)

  return s1,s2

In [68]:
# Função G(s)
print("Coeficientes do numerador de G(s): ", end = ''); s = input()
coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
g_num = Polynomial(np.flip(coef_arr))
print("Coeficientes do denominador de G(s): ", end = ''); s = input()
coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
g_den = Polynomial(np.flip(coef_arr))

# Função H(s)
print("Coeficientes do numerador de H(s): ", end = ''); s = input()
coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
h_num = Polynomial(np.flip(coef_arr))
print("Coeficientes do denominador de H(s): ", end = ''); s = input()
coef_arr = np.array(re.findall(r'-?\d*\.?\d+', s), dtype=float)
h_den = Polynomial(np.flip(coef_arr))

# Impressão das funções
print("\nG(x) = (" + str(g_num) + ")/(" + str(g_den) + ")")
print("H(x) = (" + str(h_num) + ")/(" + str(h_den) + ")\n")

# Especificação dos polos dominantes desejados
print("Como deseja informar os polos dominantes?")
print("Digite 1 para Overshoot, Tempo de Acomodação e Tolerância de Acomodação;")
print("Digite 2 para Frequência Natural e Coeficiente de Amortecimento; ou")
print("Digite 3 para informar a parte real e imaginária diretamente.")

s = input(); s_dom = calculo_polos_dominante(s)
s1 = s_dom[0]; s2 = s_dom[1]
print("\ns1 = " + str(s1)); print("s2 = " + str(s2))


Coeficientes do numerador de G(s): 1 5
Coeficientes do denominador de G(s): 1 2 1
Coeficientes do numerador de H(s): 1
Coeficientes do denominador de H(s): 1

G(x) = (5.0 + 1.0·x)/(1.0 + 2.0·x + 1.0·x²)
H(x) = (1.0)/(1.0)

Como deseja informar os polos dominantes?
Digite 1 para Overshoot, Tempo de Acomodação e Tolerância de Acomodação;
Digite 2 para Frequência Natural e Coeficiente de Amortecimento; ou
Digite 3 para informar a parte real e imaginária diretamente.
1
Overshoot(%): 10
Tempo de acomodação(s): 3
Tolerância de acomodação(%): 5
Coeficiente de amortecimento: 0.5912
Frequência natural: 1.6915 rad/s
Frequência amortecida: 1.3642

s1 = (-1+1.3642j)
s2 = (-1-1.3642j)


In [69]:
# Controlador PD

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
phiZ = sigma_angulos(s1, polos) - sigma_angulos(s1, zeros) - 180
z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))
zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

kt = phi_modulos(s1, polos)/phi_modulos(s1, zeros)
kc = kt/(g_num.coef[-1]*h_num.coef[-1])
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kd = kc; kp = kd*z
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho derivativo do controlador: " + str(round(kd, 4)))

Zero(s) do controlador projetado: -3.0
Ganho do controlador projetado: 0.1042

Ganho proporcional do controlador: -0.3126
Ganho derivativo do controlador: 0.1042


In [ ]:
# Controlador PI

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
polos = np.append(polos, 0)

phiZ = sigma_angulos(s1, polos) - sigma_angulos(s1, zeros) - 180
z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))
zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

kt = phi_modulos(s1, polos)/phi_modulos(s1, zeros)
kc = kt/(g_num.coef[-1]*h_num.coef[-1])
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kp = kc; ki = kp*z
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho integrativo do controlador: " + str(round(ki, 4)))

Zero(s) do controlador projetado: 3.4286
Ganho do controlador projetado: 7.0

Ganho proporcional do controlador: 7.0
Ganho integrativo do controlador: 24.0


In [26]:
# Controlador PID

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
polos = np.append(polos, 0)

phiZ = (sigma_angulos(s1, polos) - sigma_angulos(s1, zeros) - 180)/2
z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))
zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

kt = phi_modulos(s1, polos)/phi_modulos(s1, zeros)
kc = kt/(g_num.coef[-1]*h_num.coef[-1])
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kd = kc; kp = 2*z*kd; ki = kd*np.power(z,2);
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho integrativo do controlador: " + str(round(ki, 4)))
print("Ganho derivativo do controlador: " + str(round(kd, 4)))

Zero(s) do controlador projetado: 1.0185
Ganho do controlador projetado: 0.567

Ganho proporcional do controlador: 1.1549
Ganho integrativo do controlador: 0.5881
Ganho derivativo do controlador: 0.567
